In [ ]:
# ------------------------------------------------------------
# Transfer Learning for Image Classification (Keras 3, TF backend)
# Dataset: Cats vs Dogs (public URL via keras.utils.get_file)
# ------------------------------------------------------------
import os
import random
import numpy as np
import tensorflow as tf
import keras
from keras import layers

# -----------------------------
# 0) Reproducibility (optional)
# -----------------------------
SEED = 1337
tf.keras.utils.set_random_seed(SEED)

# -----------------------------
# 1) Download and prepare data
# -----------------------------
# Public dataset provided by Google (a filtered version of Cats vs Dogs)
DATA_URL = "https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip"
zip_path = keras.utils.get_file(
    fname="cats_and_dogs_filtered.zip",
    origin=DATA_URL,
    extract=True,
)
base_dir = os.path.join(os.path.dirname(zip_path), "cats_and_dogs_filtered")
train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "validation")

# Dataset params
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    labels="inferred",
    label_mode="int",            # use sparse labels for simplicity
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=True,
    seed=SEED,
)

val_ds = keras.utils.image_dataset_from_directory(
    val_dir,
    labels="inferred",
    label_mode="int",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=False,
    seed=SEED,
)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# Performance: cache + prefetch
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

# --------------------------------
# 2) Data augmentation & preprocessing
# --------------------------------
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
    ],
    name="data_augmentation",
)

# Use the matching preprocess_input for the chosen backbone
from keras.applications import efficientnet
preprocess_input = efficientnet.preprocess_input  # scales to expected range

# --------------------------------
# 3) Build model with pretrained backbone
# --------------------------------
# Choose a strong, lightweight backbone
base_model = efficientnet.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=IMAGE_SIZE + (3,),
)

# Phase 1: feature extractor (frozen backbone)
base_model.trainable = False

inputs = keras.Input(shape=IMAGE_SIZE + (3,), name="input_image")
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)              # important: use training=False while frozen
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs, name="cats_dogs_efficientnetB0")

# --------------------------------
# 4) Compile & train (feature extraction)
# --------------------------------
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

print("\nPhase 1: Training classification head (frozen backbone)\n")
history_fe = model.fit(
    train_ds,
    epochs=5,                        # small for a toy run; increase if you want
    validation_data=val_ds,
)

# --------------------------------
# 5) Fine-tuning: unfreeze some top layers
# --------------------------------
# Strategy: unfreeze the top N layers of the backbone
# (You can tweak N based on dataset size/overfitting)
N = 50  # unfreeze last 50 layers
for layer in base_model.layers[-N:]:
    layer.trainable = True

# Recompile with a lower LR for fine-tuning
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

# Optional: early stopping to avoid overfitting during fine-tuning
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=3,
        restore_best_weights=True,
    )
]

print("\nPhase 2: Fine-tuning top backbone layers\n")
history_ft = model.fit(
    train_ds,
    epochs=5,                     # again small for demo; increase for better results
    validation_data=val_ds,
    callbacks=callbacks,
)

# --------------------------------
# 6) Evaluate & save
# --------------------------------
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f"\nValidation accuracy: {val_acc:.3f}")

# Save the model (Keras v3 format)
model.save("cats_dogs_efficientnet.keras")
print("Saved model to cats_dogs_efficientnet.keras")

# Optional: Inference on a batch to show usage
for images, labels in val_ds.take(1):
    preds = model.predict(images)
    top = tf.argmax(preds, axis=-1).numpy()
    print("\nSample predictions:")
    for i in range(min(5, images.shape[0])):
        print(f"  True: {class_names[labels[i].numpy()]:<5} | Pred: {class_names[top[i]]}")
    break